# Supplier Intelligence Copilot — SQL Analytics

This notebook builds the structured analytics foundation for the Supplier Intelligence Copilot.

### Objectives

- Validate the SQLite procurement database
- Analyze supplier spend and concentration
- Measure SLA, delivery, defect and invoice performance
- Identify high-spend underperforming suppliers
- Detect supplier performance deterioration using SQL window functions
- Build rolling performance metrics
- Create an interpretable supplier risk ranking
- Define structured analytics that can later be consumed by the GenAI copilot

> All supplier and procurement data used in this project is synthetic.

Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name=="notebooks":
    PROJECT_ROOT=PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.database import get_connection

conn = get_connection()

def run_query(query,params=None):
    return pd.read_sql_query(query, conn, params=params)

Table Validation


In [2]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
"""

run_query(query)

,table_name
0,incidents
1,invoices
2,monthly_performance
3,supplier_reviews
4,suppliers


Spend

In [3]:
query = """
SELECT
    s.supplier_id,
    s.supplier_name,
    s.category,
    ROUND(SUM(i.invoice_amount),2) as total_spend
FROM suppliers s
JOIN invoices i
    ON s.supplier_id=i.supplier_id
GROUP BY
    s.supplier_id,
    s.supplier_name,
    s.category
ORDER BY total_spend DESC
LIMIT 10;

"""

top_spend = run_query(query)
top_spend

,supplier_id,supplier_name,category,total_spend
0,SUP081,Redwood Solutions,Cloud/SaaS,59221040.46
1,SUP012,Sterling Services,Cloud/SaaS,54272545.11
2,SUP062,SilverOak Industries,Hardware,52118450.61
3,SUP013,Atlas Global,Professional Services,46400185.82
4,SUP079,Helix Partners,Professional Services,45982690.52
5,SUP054,Sterling Industries,Telecom,45051667.69
6,SUP097,Nova Solutions,Professional Services,44509418.38
7,SUP033,Atlas Group,Professional Services,43899430.42
8,SUP064,Evergreen Systems,Professional Services,43881577.89
9,SUP039,Apex Technologies,Professional Services,43371686.31


Core KPI Summary

In [4]:
query = """
SELECT
    s.supplier_id,
    s.supplier_name,
    s.category,
    s.criticality,

    ROUND(AVG(p.sla_compliance), 2) AS avg_sla,
    ROUND(AVG(p.on_time_delivery_rate), 2) AS avg_delivery,
    ROUND(AVG(p.defect_rate), 2) AS avg_defect_rate,
    ROUND(AVG(p.invoice_accuracy), 2) AS avg_invoice_accuracy,
    SUM(p.escalation_count) AS total_escalations

FROM suppliers s
JOIN monthly_performance p
    ON s.supplier_id = p.supplier_id

GROUP BY
    s.supplier_id,
    s.supplier_name,
    s.category,
    s.criticality

ORDER BY avg_sla ASC;
"""

supplier_kpis = run_query(query)
supplier_kpis.head(15)

,supplier_id,supplier_name,category,criticality,avg_sla,avg_delivery,avg_defect_rate,avg_invoice_accuracy,total_escalations
0,SUP096,Helix Solutions,Cloud/SaaS,High,88.29,87.81,6.00,91.71,56
1,SUP024,Evergreen Services,Cloud/SaaS,High,88.55,88.23,6.07,91.63,60
2,SUP026,Prime Works,Logistics,Medium,88.67,87.76,5.82,91.90,50
3,SUP010,Vertex Industries,Logistics,Medium,88.89,87.41,5.81,91.74,59
4,SUP028,Sterling Partners,Professional Services,High,88.92,87.99,5.92,91.68,42
5,SUP056,Atlas Solutions,IT Services,High,88.93,87.68,6.03,91.86,51
6,SUP030,BluePeak Industries,Professional Services,Medium,88.94,88.43,5.79,91.80,47
7,SUP045,Crest Technologies,Marketing,Medium,89.02,87.71,5.90,91.79,62
8,SUP084,Redwood Systems,Professional Services,High,89.24,88.51,6.07,91.82,47
9,SUP063,Crest Solutions,Staffing,Low,89.37,88.52,5.87,91.91,49


Business Query

In [5]:
query = """
WITH spend AS (
    SELECT
        supplier_id,
        SUM(invoice_amount) AS total_spend
    FROM invoices
    GROUP BY supplier_id
),

performance AS (
    SELECT
        supplier_id,
        AVG(sla_compliance) AS avg_sla,
        AVG(defect_rate) AS avg_defect_rate
    FROM monthly_performance
    GROUP BY supplier_id
)

SELECT
    s.supplier_name,
    s.category,

    ROUND(sp.total_spend, 2) AS total_spend,
    ROUND(p.avg_sla, 2) AS avg_sla,
    ROUND(p.avg_defect_rate, 2) AS avg_defect_rate

FROM suppliers s
JOIN spend sp
    ON s.supplier_id = sp.supplier_id
JOIN performance p
    ON s.supplier_id = p.supplier_id

WHERE p.avg_sla < 92

ORDER BY total_spend DESC;
"""

high_spend_underperformers = run_query(query)
high_spend_underperformers.head(15)

,supplier_name,category,total_spend,avg_sla,avg_defect_rate
0,Nova Solutions,Professional Services,44509418.38,91.08,4.54
1,Atlas Group,Professional Services,43899430.42,91.97,4.70
2,Brightline Industries,Professional Services,42580998.07,91.59,3.37
3,Nimbus Partners,Professional Services,39024981.55,89.86,5.79
4,Sterling Partners,Professional Services,38448402.38,88.92,5.92
5,Vertex Systems,Cloud/SaaS,37480462.57,91.86,4.56
6,BluePeak Industries,Professional Services,36500356.44,88.94,5.79
7,Crest Works,IT Services,36247703.08,91.52,3.22
8,Helix Solutions,Cloud/SaaS,34510933.00,88.29,6.00
9,Redwood Global,Hardware,33914317.19,91.63,3.27


Invoice Quality

In [6]:
query = """
SELECT
    s.supplier_name,

    COUNT(i.invoice_id) AS invoice_count,

    ROUND(
        100.0 * SUM(i.invoice_error_flag) / COUNT(i.invoice_id),
        2
    ) AS invoice_error_rate,

    ROUND(
        AVG(i.payment_delay_days),
        2
    ) AS avg_payment_delay_days

FROM suppliers s
JOIN invoices i
    ON s.supplier_id = i.supplier_id

GROUP BY
    s.supplier_id,
    s.supplier_name

ORDER BY invoice_error_rate DESC;
"""

invoice_quality = run_query(query)
invoice_quality.head(15)

,supplier_name,invoice_count,invoice_error_rate,avg_payment_delay_days
0,Vertex Systems,167,10.78,4.88
1,Horizon Enterprises,166,10.24,4.80
2,Nexus Group,184,9.78,5.04
3,Atlas Industries,185,9.73,5.04
4,Helix Solutions,186,9.68,5.09
5,Sterling Partners,197,9.64,5.12
6,BluePeak Industries,191,9.42,4.86
7,Summit Systems,170,9.41,4.50
8,BluePeak Solutions,171,9.36,4.87
9,Sterling Systems,183,9.29,4.93


One Window Function

In [7]:
query = """
WITH monthly_change AS (

    SELECT
        supplier_id,
        month,
        sla_compliance,

        LAG(sla_compliance) OVER (
            PARTITION BY supplier_id
            ORDER BY month
        ) AS previous_sla

    FROM monthly_performance
)

SELECT
    supplier_id,
    month,
    previous_sla,
    sla_compliance AS current_sla,

    ROUND(
        sla_compliance - previous_sla,
        2
    ) AS sla_change

FROM monthly_change

WHERE
    previous_sla IS NOT NULL
    AND sla_compliance - previous_sla <= -3

ORDER BY sla_change ASC;
"""

sla_declines = run_query(query)
sla_declines.head(20)

,supplier_id,month,previous_sla,current_sla,sla_change
0,SUP075,2025-12-01,92.65,86.43,-6.22
1,SUP079,2025-03-01,94.63,89.13,-5.50
2,SUP077,2025-06-01,95.85,91.01,-4.84
3,SUP059,2025-07-01,95.55,90.78,-4.77
4,SUP023,2026-06-01,89.83,85.18,-4.65
5,SUP087,2026-03-01,93.29,88.88,-4.41
6,SUP029,2026-06-01,98.09,93.75,-4.34
7,SUP043,2026-06-01,93.22,88.88,-4.34
8,SUP024,2025-07-01,90.68,86.38,-4.30
9,SUP051,2026-05-01,89.92,85.62,-4.30


Rolling Performance

In [8]:
query = """
SELECT
    supplier_id,
    month,
    sla_compliance,

    ROUND(
        AVG(sla_compliance) OVER (
            PARTITION BY supplier_id
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS sla_3m_avg

FROM monthly_performance

ORDER BY
    supplier_id,
    month;
"""

rolling_sla = run_query(query)
rolling_sla.head(20)

,supplier_id,month,sla_compliance,sla_3m_avg
0,SUP001,2025-01-01,89.42,89.42
1,SUP001,2025-02-01,91.79,90.61
2,SUP001,2025-03-01,88.86,90.02
3,SUP001,2025-04-01,90.62,90.42
4,SUP001,2025-05-01,89.39,89.62
5,SUP001,2025-06-01,89.75,89.92
6,SUP001,2025-07-01,88.95,89.36
7,SUP001,2025-08-01,88.62,89.11
8,SUP001,2025-09-01,87.69,88.42
9,SUP001,2025-10-01,89.71,88.67


Combined Intelligence Table

In [9]:
query = """
WITH spend AS (

    SELECT
        supplier_id,
        SUM(invoice_amount) AS total_spend,
        100.0 * SUM(invoice_error_flag) / COUNT(*) AS invoice_error_rate
    FROM invoices
    GROUP BY supplier_id
),

performance AS (

    SELECT
        supplier_id,
        AVG(sla_compliance) AS avg_sla,
        AVG(on_time_delivery_rate) AS avg_delivery,
        AVG(defect_rate) AS avg_defect,
        SUM(escalation_count) AS escalations
    FROM monthly_performance
    GROUP BY supplier_id
),

incident_summary AS (

    SELECT
        supplier_id,
        COUNT(*) AS incidents
    FROM incidents
    GROUP BY supplier_id
)

SELECT
    s.supplier_id,
    s.supplier_name,
    s.category,
    s.criticality,

    ROUND(sp.total_spend, 2) AS total_spend,
    ROUND(p.avg_sla, 2) AS avg_sla,
    ROUND(p.avg_delivery, 2) AS avg_delivery,
    ROUND(p.avg_defect, 2) AS avg_defect,
    ROUND(sp.invoice_error_rate, 2) AS invoice_error_rate,

    p.escalations,
    COALESCE(i.incidents, 0) AS incidents

FROM suppliers s

LEFT JOIN spend sp
    ON s.supplier_id = sp.supplier_id

LEFT JOIN performance p
    ON s.supplier_id = p.supplier_id

LEFT JOIN incident_summary i
    ON s.supplier_id = i.supplier_id

ORDER BY avg_sla ASC;
"""

supplier_intelligence = run_query(query)
supplier_intelligence.head(20)

,supplier_id,supplier_name,category,criticality,total_spend,avg_sla,avg_delivery,avg_defect,invoice_error_rate,escalations,incidents
0,SUP096,Helix Solutions,Cloud/SaaS,High,34510933.00,88.29,87.81,6.00,9.68,56,40
1,SUP024,Evergreen Services,Cloud/SaaS,High,33502043.53,88.55,88.23,6.07,7.73,60,36
2,SUP026,Prime Works,Logistics,Medium,20568889.89,88.67,87.76,5.82,6.25,50,37
3,SUP010,Vertex Industries,Logistics,Medium,17036279.49,88.89,87.41,5.81,7.78,59,41
4,SUP028,Sterling Partners,Professional Services,High,38448402.38,88.92,87.99,5.92,9.64,42,31
5,SUP056,Atlas Solutions,IT Services,High,29551763.00,88.93,87.68,6.03,9.04,51,28
6,SUP030,BluePeak Industries,Professional Services,Medium,36500356.44,88.94,88.43,5.79,9.42,47,30
7,SUP045,Crest Technologies,Marketing,Medium,25641802.52,89.02,87.71,5.90,6.63,62,33
8,SUP084,Redwood Systems,Professional Services,High,33153328.45,89.24,88.51,6.07,6.40,47,35
9,SUP063,Crest Solutions,Staffing,Low,19527402.34,89.37,88.52,5.87,5.92,49,43
